# Data Ingestion & Treatment

Notebook unificado de ingestão e tratamento de dados para o projeto Energy Poverty.

Todos os ficheiros de dados são lidos a partir da pasta `data/` neste directório.

**Datasets produzidos:**
- `df_household_income_raw` — Household Income por município 2015–2021 (INE API)
- `df_household_income_forecast` — Projeção Household Income 2015–2026
- `df_energy_household` — Consumo de energia doméstica por município/ano (Carlota CSV)
- `df_energy_prices` — Preços de energia elétrica por ano/banda (Simão)
- `df_energy_efficiency` — Certificados energéticos por concelho/ano (Madalena)
- `df_median_age` — Idade mediana por município 2023
- `df_energy_expenditure` — Gasto energético anual por CPE (energia × preço)

## 0. Imports e configuração de paths

In [ ]:
import requests
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

DATA_DIR = Path("data")

---
# 1. Household Income

Fonte: INE — Rendimento médio disponível por habitante (varcd=0009762)  
https://www.ine.pt/xportal/xmain?xpid=INE&xpgid=ine_indicadores&indOcorrCod=0009762&contexto=bd&selTab=tab2

## 1.1 Recolha de dados via API INE (2015–2021)

In [ ]:
anos = range(2015, 2022)
rows = []

for ano in anos:
    url = (
        f"https://www.ine.pt/ine/json_indicador/pindica.jsp"
        f"?op=2&varcd=0009762&lang=PT&Dim1=S7A{ano}"
    )
    data = requests.get(url).json()

    for entry in data:
        # Cada entry tem uma chave "Dados" com os registos desse ano
        for ano_key, registos in entry.get("Dados", {}).items():
            for r in registos:
                geocod = r.get("geocod", "")
                # Excluir agregados (Portugal, regiões) — só municípios têm geocod numérico
                if not geocod or not geocod[0].isdigit():
                    continue
                rows.append({
                    "ANO":                    int(ano_key),
                    "Município":              r.get("geodsg"),
                    "Household Income (EUR)": pd.to_numeric(r.get("valor"), errors="coerce"),
                })

# Construir o DataFrame final, remover duplicados e ordenar
df_household_income_raw = (
    pd.DataFrame(rows)
    .dropna(subset=["Household Income (EUR)"])
    .drop_duplicates()
    .sort_values(["ANO", "Município"])
    .reset_index(drop=True)
)

print(f"{len(df_household_income_raw)} registos | {df_household_income_raw['Município'].nunique()} municípios | Anos: {sorted(df_household_income_raw['ANO'].unique())}")
df_household_income_raw

In [ ]:
#  Média do rendimento por ano (anos com maior rendimento médio) 
df_household_income_raw.groupby("ANO")["Household Income (EUR)"].mean().round(2).sort_values(ascending=False).reset_index()

In [ ]:
#  Top 10 registos com maior rendimento (ANO + Município, overall) 
df_household_income_raw.nlargest(10, "Household Income (EUR)").reset_index(drop=True)

In [ ]:
#  Todos os registos ordenados por rendimento decrescente 
df_household_income_raw.sort_values("Household Income (EUR)", ascending=False).reset_index(drop=True)

In [ ]:
df_household_income_raw.to_parquet(DATA_DIR / "household-income-15-21.parquet")

## 1.2 Previsão de Household Income por município (até 2026)

**Objetivo**: a partir do ficheiro `household-income-15-21.parquet`, projetar valores de *household income* para 2022–2026, município a município.

O crescimento anual de cada município é calculado com base em três componentes com pesos fixos:
- **50%** – CAGR histórico do próprio município (crescimento pré-COVID, até 2019)
- **20%** – Fatores macroeconómicos: inflação e crescimento do PIB (10% cada)
- **30%** – Crescimento salarial: salário mínimo e salário médio (15% cada)

**Nota**: este notebook foi gerado em 2026-05-16.

### Fontes (para defender o trabalho)

Não é feito forecast da inflação/salário mínimo; são usadas séries publicadas.

- Salário mínimo Mensal nacional (tabela anual, inclui 2026): https://www.pordata.pt/portugal/evolucao+do+salario+minimo+nacional-74

- Salário Médio Líquido Mensal Em portugal - https://pt.tradingeconomics.com/portugal/wages

- Previsão macro (PIB) – PORDATA - https://www.pordata.pt/portugal/taxa+de+crescimento+do+pib-2298
    - Estimativa para 2026 https://www.bportugal.pt/publicacao/boletim-economico-marco-2026

-Inflação segundo Banco de Portugal - https://bpstat.bportugal.pt/serie/12704650
    - Estimativa para 2026: https://pt.tradingeconomics.com/portugal/inflation-cpi


- (Proxy para mediana) 'Rendimento mediano disponível' na PORDATA Retratos da Europa: https://retratos.europa.pordata.pt/custo-de-vida/portugal

### 1.2.0 Análise de municípios: Ler o Dataset e normalizar colunas

De parquet para dataframe e normalização de colunas para evitar erros no pipeline.

In [ ]:
# De parquet para dataframe.
df_income_forecast = pd.read_parquet(DATA_DIR / 'household-income-15-21.parquet')



# Mapeamento de nomes alternativos para year/municipio/income.
rename_map = {
    'ANO': 'year',
    'Município': 'municipio',
    'Household Income (EUR)': 'income',
}

# Intera-se entre pares do rename_map para renomear colunas correspondentes.
for k, v in rename_map.items():
    if k in df_income_forecast.columns:
        df_income_forecast = df_income_forecast.rename(columns={k: v})

# Conversão de tipos.
df_income_forecast['year'] = df_income_forecast['year'].astype(int)
df_income_forecast['municipio'] = df_income_forecast['municipio'].astype(str)
df_income_forecast['income'] = pd.to_numeric(df_income_forecast['income'], errors='coerce')

# Ordenação por município e ano para cálculo temporal subsequente.
df_income_forecast = df_income_forecast.sort_values(['municipio', 'year']).reset_index(drop=True)

df_income_forecast.head(8)

### 1.2.1 Calcular CAGR pré-COVID (até 2019)

CAGR significa Compound Annual Growth Rate. O objetivo é calcular o crescimento anual composto de cada município até 2019, evitando o efeito do choque COVID.

A lógica é fixar o município e medir a variação entre os pontos inicial e final do período pré-COVID. Por exemplo, se o income passa de 100 para 121 em 2 anos, o CAGR é (121/100)^(1/2) - 1 = 0,10, ou 10% ao ano.

In [ ]:
def calculate_pre_covid_cagr(group: pd.DataFrame) -> float:
    """Calculate CAGR for one municipio using observations up to 2019."""
    """
    Args:
        group: DataFrame slice for a single municipio (all years for that municipio).
               The caller uses `df_income_forecast.groupby('municipio')`, so each `group` contains only one municipio,
               e.g. rows for 'Abrantes' from 2015..2019. This function will further filter to years <= 2019.

    Returns:
        CAGR as float (e.g. 0.10 for 10%/year) or np.nan if not computable.
    """
    # Filtra até 2019 para estimar crescimento estrutural sem o efeito COVID.
    g = group[group['year'] <= 2019].copy()

    # São necessários pelo menos dois pontos no tempo para calcular CAGR.
    # Exemplo: para Abrantes pode haver linhas para 2015,2016,2017,2018,2019 -> shape[0] >= 2 ok.
    if g.shape[0] < 2:
        return np.nan

    # Pegamos o primeiro e o último valor observados no período filtrado (mesmo municipio).
    start = g.iloc[0]['income']
    end = g.iloc[-1]['income']
    # Número de anos entre os dois pontos (p.ex. 2019 - 2015 = 4).
    years = g.iloc[-1]['year'] - g.iloc[0]['year']



    # Fórmula do CAGR: (end/start)^(1/years) - 1.
    # Exemplo: start=100, end=121, years=2 => (121/100)^(1/2) - 1 = 0.10 (10% ao ano).
    return (end / start) ** (1 / years) - 1

cagr_df = (
    df_income_forecast.groupby('municipio', as_index=False)
      .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))
)

cagr_df.head()

### 1.2.2 Tabela Macro com fatores económicos para extrapolar os Median Household Income (inflação, PIB, salários)

Dados com bases citadas acima.

- O ambiente do notebook pode não ter acesso à Internet, por isso as séries são definidas manualmente e validadas.
- É verificado que todos os anos necessários estão presentes para evitar buracos no modelo.

**Como preencher**: copiar os valores dos sites e substituir nos dicionários abaixo.

In [ ]:
YEARS_FORECAST = [2022, 2023, 2024, 2025, 2026]

# Inflação anual (decimal). Fonte: INE / Banco de Portugal.
inflation = {2022: 0.0783, 2023: 0.0431, 2024: 0.0242, 2025: 0.0234, 2026: 0.0330}

# Crescimento real do PIB (decimal). Fonte: Banco de Portugal / OCDE.
gdp_growth = {2022: 0.070, 2023: 0.031, 2024: 0.022, 2025: 0.019, 2026: 0.018}

# Salário mínimo nacional (EUR/mês). Fonte: PORDATA.
min_wage = {2022: 705, 2023: 760, 2024: 820, 2025: 870, 2026: 920}

# Salário médio líquido (EUR/mês). Fonte: Trading Economics / PORDATA.
avg_wage = {2022: 1010, 2023: 1090, 2024: 1230, 2025: 1310, 2026: 1370}

# Construir tabela macro
macro = pd.DataFrame({
    'year': YEARS_FORECAST,
    'inflation': [inflation[y] for y in YEARS_FORECAST],
    'gdp_growth': [gdp_growth[y] for y in YEARS_FORECAST],
    'min_wage':   [min_wage[y]   for y in YEARS_FORECAST],
    'avg_wage':   [avg_wage[y]   for y in YEARS_FORECAST],
})

# Calcular taxas de crescimento anuais dos salários
macro['g_min_wage'] = macro['min_wage'].pct_change().fillna(0)
macro['g_avg_wage'] = macro['avg_wage'].pct_change().fillna(0)

macro

### 1.2.3 Preparar base (último ano observado) e juntar CAGR

O ponto de partida para a projeção é o income de cada município em 2021 (último ano observado), ao qual se junta o CAGR calculado na secção anterior.

### 1.2.4 Loop de projeção (2022–2026)

Fórmula de crescimento anual com pesos fixos:

```
g_total = 0.50 * cagr_pre_covid
        + 0.10 * inflação
        + 0.10 * crescimento_pib
        + 0.15 * crescimento_salario_minimo
        + 0.15 * crescimento_salario_medio
```

A projeção aplica este `g_total` ao income do ano anterior, iterando de 2022 a 2026.

In [ ]:
latest_year = int(df_income_forecast['year'].max())

base_df = df_income_forecast[df_income_forecast['year'] == latest_year].copy()
base_df = base_df.merge(cagr_df, on='municipio', how='left')

base_df.head()

In [ ]:
projections = []
current_df = base_df.copy()

for _, row in macro.sort_values('year').iterrows():
    year = int(row['year'])

    next_df = current_df.copy()

    # Taxa de crescimento anual = média ponderada dos cinco fatores
    g_total = (
        0.50 * next_df['cagr_pre_covid'] +  # 50% tendência local pré-COVID
        0.10 * float(row['inflation'])    +  # 10% inflação
        0.10 * float(row['gdp_growth'])   +  # 10% crescimento do PIB
        0.15 * float(row['g_min_wage'])   +  # 15% crescimento do salário mínimo
        0.15 * float(row['g_avg_wage'])      # 15% crescimento do salário médio
    )

    next_df['income'] = next_df['income'] * (1 + g_total)
    next_df['year'] = year

    projections.append(next_df[['municipio', 'year', 'income']])
    current_df = next_df

projection_df = pd.concat(projections, ignore_index=True)

df_household_income_forecast = pd.concat(
    [df_income_forecast[['municipio', 'year', 'income']], projection_df],
    ignore_index=True
).sort_values(['municipio', 'year']).reset_index(drop=True)



df_household_income_forecast.head(20)

### 1.2.5 Sanity checks rápidos

Top 10 municípios em 2026 e série temporal de Lisboa para validar a projeção.

In [ ]:
# Top 10 municípios em 2026
df_household_income_forecast[df_household_income_forecast['year'] == 2026].sort_values('income', ascending=False).head(10)

In [ ]:
df_household_income_forecast[df_household_income_forecast['municipio'] == 'Lisboa'].sort_values('year')

In [ ]:
df_household_income_forecast.set_index("year").to_parquet(DATA_DIR / 'household_income_projection_2026.parquet', index=True)

---
# 2. Carlota — CPE e Consumo de Energia

## 2.1 CPE and Energy Consumption — CSV Data Loading

### Normalize column names for easier use

In [ ]:
df_cpe = pd.read_csv(
    DATA_DIR / "20-caracterizacao-pes-contrato-ativo.csv",
    sep=";",
    encoding="utf-8-sig"
)

df_consumption = pd.read_csv(
    DATA_DIR / "3-consumos-faturados-por-municipio-ultimos-10-anos.csv",
    sep=";",
    encoding="utf-8-sig"
)

In [ ]:
# Rename columns to match original API field names used in the analysis
df_cpe.columns = [c.strip() for c in df_cpe.columns]
df_consumption.columns = [c.strip() for c in df_consumption.columns]

df_cpe = df_cpe.rename(columns={
    "Ano": "ano",
    "Mês": "mes",
    "Data": "data",
    "Distrito": "distrito",
    "Concelho": "concelho",
    "Freguesia": "freguesia",
    "Tipo de Instalação": "tipo_de_instalacao",
    "Nível de Tensão": "nivel_de_tensao",
    "cpes": "cpes",
    "CodDistrito": "coddistrito",
    "CodDistritoConcelho": "coddistritoconcelho",
    "coddistritoconcelhofreguesia": "coddistritoconcelhofreguesia"
})

df_consumption = df_consumption.rename(columns={
    "Ano": "ano",
    "Mês": "mes",
    "Data": "data",
    "Distrito": "distrito",
    "Concelho": "concelho",
    "Freguesia": "freguesia",
    "Nível de Tensão": "nivel_de_tensao",
    "Energia Ativa": "energia_ativa_kwh",
    "coddistrito": "coddistrito",
    "coddistritoconcelho": "coddistritoconcelho",
    "coddistritoconcelhofreguesia": "coddistritoconcelhofreguesia",
    "mes_int": "mes_int"
})

### Preview the data

In [ ]:
print(df_cpe.head())

In [ ]:
print(df_consumption.head())

### Check for missing values

In [ ]:
print(df_consumption.isnull().sum())
print(df_cpe.isnull().sum())

### Inspect column names and unique values

In [ ]:
print(df_consumption.columns.tolist())
print(df_cpe.columns.tolist())

In [ ]:
print(df_consumption["nivel_de_tensao"].unique()[:20])
print(df_cpe["nivel_de_tensao"].unique()[:20])

In [ ]:
print(df_consumption["freguesia"].unique()[:20])
print(df_cpe["freguesia"].unique()[:20])

In [ ]:
print(df_consumption["concelho"].unique()[:20])
print(df_cpe["concelho"].unique()[:20])

### Filter for low voltage (Baixa Tensão)

In [ ]:
# Keep only Baixa Tensão in consumption
df_consumption_bt = df_consumption[
    df_consumption["nivel_de_tensao"] == "Baixa Tensão"
].copy()

# Keep only BTN and BTE in CPE
df_cpe_bt = df_cpe[
    df_cpe["nivel_de_tensao"].isin([
        "Baixa Tensão Normal",
        "Baixa Tensão Especial"
    ])
].copy()

### Keep only December CPE records (snapshot per year)

In [ ]:
df_cpe_dec = df_cpe_bt[df_cpe_bt["mes"] == 12].copy()

### Aggregate by year, district and municipality

In [ ]:
cpe_year = (
    df_cpe_dec
    .groupby(["ano", "distrito", "concelho"], as_index=False)["cpes"]
    .sum()
)

In [ ]:
consumption_year = (
    df_consumption_bt
    .groupby(["ano", "distrito", "concelho"], as_index=False)["energia_ativa_kwh"]
    .sum()
)

### Merge and compute kWh per CPE

In [ ]:
df_energy_household = consumption_year.merge(
    cpe_year,
    on=["ano", "distrito", "concelho"],
    how="inner"
)

df_energy_household["kwh_por_cpe"] = (
    df_energy_household["energia_ativa_kwh"] / df_energy_household["cpes"]
)

In [ ]:
resultado_carlota = df_energy_household[
    ["ano", "distrito", "concelho", "energia_ativa_kwh", "cpes", "kwh_por_cpe"]
]

print(resultado_carlota.head())
print(resultado_carlota.shape)

In [ ]:
counts_carlota = (
    df_energy_household.groupby('ano')['concelho']
      .nunique()
      .reset_index(name='num_concelhos')
)

print(counts_carlota)

### Diagnostic: check overlapping years between datasets

In [ ]:
set(consumption_year["ano"]).intersection(set(cpe_year["ano"]))

In [ ]:
consumption_year.merge(cpe_year, on=["ano","distrito","concelho"], how="inner").shape

In [ ]:
print(sorted(df_consumption["ano"].unique()))
print(sorted(df_cpe["ano"].unique()))

In [ ]:
df_consumption_year_carlota = (
    df_consumption[df_consumption["nivel_de_tensao"] == "Baixa Tensão"]
    .groupby(["ano"], as_index=False)["energia_ativa_kwh"]
    .sum()
)

df_cpe_year_carlota = (
    df_cpe[df_cpe["nivel_de_tensao"].isin([
        "Baixa Tensão Normal",
        "Baixa Tensão Especial"
    ]) & (df_cpe["mes"] == 12)]
    .groupby(["ano"], as_index=False)["cpes"]
    .sum()
)

print(df_consumption_year_carlota)
print(df_cpe_year_carlota)

In [ ]:
df_consumption_year_carlota.merge(df_cpe_year_carlota, on="ano", how="inner")

In [ ]:
print("ANO:")
print(df_cpe["ano"].unique())

print("\nMES:")
print(df_cpe["mes"].unique())

print("\nDATA:")
print(df_cpe["data"].unique()[:20])

---
# 3. Energy Prices (Simão)

In [ ]:
energy_prices_df = pd.read_excel(DATA_DIR / "dgeg-ped-1985-2025-s2_pt.xlsx")

In [ ]:
energy_prices_df

In [ ]:
print(energy_prices_df.columns.tolist())

In [ ]:
print(energy_prices_df.isnull().sum())

In [ ]:
#The portugal data comes in a seperate sheet and 15 lines deep
energy_prices_df2 = pd.read_excel(
    DATA_DIR / "dgeg-ped-1985-2025-s2_pt.xlsx",
    sheet_name="Preços Médios em Portugal",
    skiprows=15
)

In [ ]:
energy_prices_df2

In [ ]:
energy_prices_df2.columns = [
    "Banda",
    "Consumo_Min",
    "Consumo_Max",
    "Preco_Sem_Impostos",
    "Preco_Sem_IVA",
    "Preco_Final"
]

print(energy_prices_df2)

In [ ]:
print(energy_prices_df2.isnull().sum())

In [ ]:
energy_prices_df2.shape

In [ ]:
energy_prices_df2.head(10)

In [ ]:
#Toremove the lines that contain just titles or blank lines
clean_prices_df = energy_prices_df2[energy_prices_df2["Banda"].astype(str).str.contains("Banda -")]
clean_prices_df.head(10)

In [ ]:
year = 2025
years = []
for i in range(len(clean_prices_df)):
    years.append(year)
    if(i + 1)%10== 0:
        year -= 1

clean_prices_df["Ano"]= years

In [ ]:
clean_prices_df.head(20)

In [ ]:
clean_prices_df = clean_prices_df.drop(columns = ["Preco_Sem_IVA", "Preco_Sem_Impostos"])
clean_prices_df

In [ ]:
clean_prices_df["Consumo_Max"] = clean_prices_df["Consumo_Max"].fillna("Sem limite")

In [ ]:
clean_prices_df = clean_prices_df.sort_values("Ano", ascending = False)
clean_prices_df

In [ ]:
clean_prices_df_da = (
    clean_prices_df
    .reset_index()
    .query("Banda == 'Banda - DC'")#referenciar estatisticas da erce
)

In [ ]:
clean_prices_df_da = clean_prices_df_da.drop(columns = ['Consumo_Min','Consumo_Max'])

In [ ]:
clean_prices_df_da

In [ ]:
df_energy_prices = clean_prices_df_da.copy()
df_energy_prices.to_parquet(DATA_DIR / "energy_prices.parquet")

---
# 4. Energy Efficiency (Madalena)

In [ ]:
df_eff_2015 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2015.csv', encoding='utf-16')
df_eff_2016 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2016.csv', encoding='utf-16')
df_eff_2017 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2017.csv', encoding='utf-16')
df_eff_2018 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2018.csv', encoding='utf-16')
df_eff_2019 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2019.csv', encoding='utf-16')
df_eff_2020 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2020.csv', encoding='utf-16')
df_eff_2021 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2021.csv', encoding='utf-16')
df_eff_2022 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2022.csv', encoding='utf-16')
df_eff_2023 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2023.csv', encoding='utf-16')
df_eff_2024Q1 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q1.csv', encoding='utf-16')
df_eff_2024Q2 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q2.csv', encoding='utf-16')
df_eff_2024Q3 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q3.csv', encoding='utf-16')
df_eff_2024Q4 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q4.csv', encoding='utf-16')
df_eff_2025Q1 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q1.csv', encoding='utf-16')
df_eff_2025Q2 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q2.csv', encoding='utf-16')
df_eff_2025Q3 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q3.csv', encoding='utf-16')
df_eff_2025Q4 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q4.csv', encoding='utf-16')

In [ ]:
df_eff_2015.shape

In [ ]:
df_eff_2016.shape

In [ ]:
df_eff_2017.shape

In [ ]:
df_eff_2018.shape

In [ ]:
df_eff_2019.shape

In [ ]:
df_eff_2020.shape

In [ ]:
df_eff_2021.shape

In [ ]:
df_eff_2022.shape

In [ ]:
df_eff_2023.shape

In [ ]:
df_eff_2023.shape

In [ ]:
df_eff_2024Q1.shape

In [ ]:
df_eff_2024Q2.shape

In [ ]:
df_eff_2024Q3.shape

In [ ]:
df_eff_2024Q4.shape

In [ ]:
df_eff_2025Q1.shape

In [ ]:
df_eff_2025Q2.shape

In [ ]:
df_eff_2025Q3.shape

In [ ]:
df_eff_2025Q4.shape

In [ ]:
df_eff_2015.columns

In [ ]:
df_eff_2016.columns

In [ ]:
df_eff_2017.columns

In [ ]:
df_eff_2018.columns

In [ ]:
df_eff_2019.columns

In [ ]:
df_eff_2020.columns

In [ ]:
df_eff_2021.columns

In [ ]:
df_eff_2022.columns

In [ ]:
df_eff_2023.columns

In [ ]:
df_eff_2024Q1.columns

In [ ]:
df_eff_2024Q2.columns

In [ ]:
df_eff_2024Q3.columns

In [ ]:
df_eff_2024Q4.columns

In [ ]:
df_eff_2025Q1.columns

In [ ]:
df_eff_2025Q2.columns

In [ ]:
df_eff_2025Q3.columns

In [ ]:
df_eff_2025Q4.columns

In [ ]:
df_eff_2015_2025 = pd.concat([
    df_eff_2015, df_eff_2016, df_eff_2017, df_eff_2018, df_eff_2019,
    df_eff_2020, df_eff_2021, df_eff_2022, df_eff_2023,
    df_eff_2024Q1, df_eff_2024Q2, df_eff_2024Q3, df_eff_2024Q4,
    df_eff_2025Q1, df_eff_2025Q2, df_eff_2025Q3, df_eff_2025Q4
], ignore_index=True)

In [ ]:
df_eff_2015_2025.columns

In [ ]:
df_eff_2015_2025.shape

In [ ]:
df_eff_2015_2025.head()

In [ ]:
df_eff_2015_2025.isnull().sum()

In [ ]:
df_eff_2015_2025['Tipo de Doc / Contexto'].unique()

In [ ]:
df_eff_2015_2025['Tipo de Edifício'].unique()

We are going to eliminate values equal to "Serviços" in column "Tipo de Edifício". Energy poverty has to do with habitational buildings only.

In [ ]:
df_eff_clean = df_eff_2015_2025.drop(df_eff_2015_2025[df_eff_2015_2025['Tipo de Edifício'] == 'Serviços'].index)

In [ ]:
df_eff_clean.shape

In [ ]:
df_eff_clean['Tipo de Edifício'].unique()

In [ ]:
df_eff_clean['Classe Energética'].unique()

We dropped coluns "Total de Emissões CO2" and "Área Média Útil de Pavimento", because those are not useful for energy poverty assessment.

In [ ]:
df_eff_clean = df_eff_clean.drop(columns=['Total de Emissões CO2','Área Média Útil de Pavimento'])

In [ ]:
df_eff_clean.columns

In [ ]:
df_eff_clean['Tipo de Doc / Contexto'].value_counts()

A big part of the buildings are existing buildings, which means that, in our dataframe, we have certificates of new buildings and existing buildings.
We deleted the rows about the certificate of buildings still at the project stage because these buildings do not exist yet.

In [ ]:
df_eff_clean = df_eff_clean.drop(df_eff_clean[df_eff_clean['Tipo de Doc / Contexto'] == 'Projeto Edifícios Novos'].index)
df_eff_clean = df_eff_clean.drop(df_eff_clean[df_eff_clean['Tipo de Doc / Contexto'] == 'Projeto Renovação Edifícios'].index)

In [ ]:
df_eff_clean['Tipo de Doc / Contexto'].value_counts()

In [ ]:
df_eff_clean.shape

In [ ]:
df_eff_clean.head(20)

In [ ]:
df_eff_clean['Tipo de Edifício'].value_counts()

Now that all building types, are the same, we can erase this column.

In [ ]:
df_eff_clean = df_eff_clean.drop(columns=['Tipo de Edifício'])

In [ ]:
df_eff_clean["Mês de Emissão"].dtype

We want the dataframe to be by year and not by month.

In [ ]:
df_eff_clean["Mês de Emissão"] = pd.to_datetime(df_eff_clean["Mês de Emissão"])

In [ ]:
df_eff_clean["Mês de Emissão"] = df_eff_clean["Mês de Emissão"].dt.year

In [ ]:
df_energy_efficiency = (
    df_eff_clean.groupby(
        ['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho', 'Tipo de Doc / Contexto', 'Classe Energética'],
        as_index=False
    )['Total de Certificados Emitidos'].sum()
)

In [ ]:
df_eff_clean.head(100)

In [ ]:
df_energy_efficiency.to_parquet(DATA_DIR / "Energy_eff.parquet")

---
# 5. Median Age

In [ ]:
df_median_age = pd.read_excel(
    DATA_DIR / "idade_mediana_por_municipio_1.xlsx"
)

In [ ]:
df_median_age

---
# 6. Uniformização de nomes e cálculo do gasto energético

We are going to multiply the energy prices for the energy consumption to obtain the annual energy expenditure per cpe.

In [ ]:
df_energy_expenditure = df_energy_household.merge(
    df_energy_prices[['Ano', 'Preco_Final']],
    left_on='ano',
    right_on='Ano',
    how='left'
)

In [ ]:
df_energy_expenditure = df_energy_expenditure.drop(columns=['Ano'])

In [ ]:
df_energy_expenditure['annual_expenditure'] = (
    df_energy_expenditure['kwh_por_cpe'] * df_energy_expenditure['Preco_Final']
)

In [ ]:
df_energy_expenditure

## 6.1 Uniformização de nomes de concelhos e distritos

We are going to uniformize the names of the portuguese concelhos and districts in the different data frames

In [ ]:
df_municipios_distritos = pd.read_parquet(DATA_DIR / "household_income_projection_2026.parquet").reset_index()
# Extrair lista única de municípios a partir da projeção de income
df_municipios_distritos = (
    df_household_income_forecast[['municipio']]
    .drop_duplicates()
    .rename(columns={'municipio': 'municipio'})
    .assign(distrito_ou_regiao_autonoma=None)
)

First, we imported the dataframe which will be the source of truth for the names of municipios and districts.

In [ ]:
df_municipios_distritos.isnull().values.any()

In [ ]:
df_energy_expenditure.isnull().values.any()

In [ ]:
df_energy_efficiency.isnull().values.any()

In [ ]:
df_household_income_forecast.isnull().values.any()

In [ ]:
df_median_age.isnull().values.any()

In [ ]:
def normalize(column):

    normalized_values = []

    for x in column:
        x = str(x).strip().lower() #converts text to lowercase

        x= unicodedata.normalize("NFKD", x)
        x = "".join(c for c in x if not unicodedata.combining(c)) #removes accents

        x = x.replace("-", " ") #deletes hiphens
        x = " ".join(x.split()) #deletes extra spaces

        normalized_values.append(x)

    return normalized_values

In [ ]:
df_municipios_distritos["municipio_limpo"] = normalize(df_municipios_distritos["municipio"])
df_energy_expenditure["concelho_limpo"] = normalize(df_energy_expenditure["concelho"])
df_median_age["Concelho_limpo"] = normalize(df_median_age["Concelho"])
df_household_income_forecast["municipio_limpo"] = normalize(df_household_income_forecast["municipio"])
df_energy_expenditure["distrito_limpo"] = normalize(df_energy_expenditure["distrito"])
df_median_age["Distrito_limpo"] = normalize(df_median_age["Distrito"])

We applied the normalize function to all the concellho/municipio columns of our dataframes.

In [ ]:
municipios_ref = df_municipios_distritos[
    ["municipio_limpo", "municipio"]
]

---
# 7. Datasets finais

Sumário dos datasets produzidos e disponíveis para análise posterior.

In [ ]:
print("=== Datasets disponíveis ===")
print(f"df_household_income_raw       : {df_household_income_raw.shape}  — Household Income por município 2015–2021")
print(f"df_household_income_forecast  : {df_household_income_forecast.shape}  — Projeção Household Income 2015–2026")
print(f"df_energy_household           : {df_energy_household.shape}  — Consumo doméstico kWh/CPE por concelho/ano")
print(f"df_energy_prices              : {df_energy_prices.shape}  — Preços energia elétrica Banda DC por ano")
print(f"df_energy_efficiency          : {df_energy_efficiency.shape}  — Certificados energéticos por concelho/ano")
print(f"df_median_age                 : {df_median_age.shape}  — Idade mediana por município 2023")
print(f"df_energy_expenditure         : {df_energy_expenditure.shape}  — Gasto energético anual por CPE")